# Notebook 3: City-Specific Pollutant Correlation

This notebook computes city-wise correlations for the top AQI pollutant (from Notebook 1) against weather and demographic factors, then exports a JSON artifact for backend use.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACT_DIR = PROJECT_ROOT / 'backend' / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def read_csv_robust(path):
    for enc in ('utf-8', 'utf-8-sig', 'cp1252', 'latin1'):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding_errors='replace')

aqi_df = pd.read_csv(DATA_DIR / 'aqi_data.csv').rename(columns={'S02': 'SO2'})
weather_df = pd.read_csv(DATA_DIR / 'weather.csv')
demo_df = pd.read_csv(DATA_DIR / 'demographics.csv')
date_df = pd.read_csv(DATA_DIR / 'date_dimension.csv')
cities_df = read_csv_robust(DATA_DIR / 'cities.csv')

In [3]:
with open(ARTIFACT_DIR / 'feature_importance.json', 'r', encoding='utf-8') as fp:
    feature_importance = json.load(fp)

top_pollutant = feature_importance['top_pollutant']
top_pollutant

'PM2.5'

In [4]:
joined = (
    aqi_df
    .merge(weather_df, on=['city_id', 'date_key'], how='left')
    .merge(date_df[['date_key', 'year', 'month', 'month_number']], on='date_key', how='left')
    .merge(demo_df, on=['city_id', 'year'], how='left')
    .merge(cities_df[['city_id', 'city_name']], on='city_id', how='left')
)

factor_cols = [
    'Temperature', 'Humidity', 'Wind Speed', 'Rainfall', 'Visibility',
    'Population', 'Population Density', 'Urbanization %', 'Vehicle Count', 'Industrial Area %'
]

for col in [top_pollutant] + factor_cols:
    joined[col] = pd.to_numeric(joined[col], errors='coerce')

joined[['city_id', 'city_name', top_pollutant] + factor_cols].head()

,city_id,city_name,PM2.5,Temperature,Humidity,Wind Speed,Rainfall,Visibility,Population,Population Density,Urbanization %,Vehicle Count,Industrial Area %
0,6,Adams,8.24,13,64.6,15.8,69.1,10000,207227,1554,81.7,120191,10.9
1,41,Alliance,8.24,12,62.2,15.2,64.0,10000,188811,1416,81.7,109510,10.9
2,71,"Ann Arbor, MI",8.24,13,63.2,15.4,66.2,10000,187326,1404,81.7,108649,10.9
3,126,Avon,8.24,14,67.1,16.8,74.5,10000,201881,1514,81.7,117090,10.9
4,181,Bayside,8.24,12,61.4,14.9,62.3,10000,199801,1498,81.7,115884,10.9


In [5]:
city_correlations = {}

for city_id, group in joined.groupby('city_id'):
    corr_df = group[[top_pollutant] + factor_cols].dropna(how='all')
    if corr_df[top_pollutant].notna().sum() < 8:
        continue

    corr_series = corr_df.corr(numeric_only=True)[top_pollutant].drop(labels=[top_pollutant]).dropna()
    if corr_series.empty:
        continue

    positive = corr_series[corr_series > 0].sort_values(ascending=False)
    negative = corr_series[corr_series < 0].sort_values()

    city_name = group['city_name'].dropna().iloc[0] if group['city_name'].notna().any() else f'City {city_id}'

    city_correlations[str(int(city_id))] = {
        'city_id': int(city_id),
        'city_name': city_name,
        'selected_pollutant': top_pollutant,
        'positive_influence': [
            {'factor': k, 'correlation': round(float(v), 5)} for k, v in positive.items()
        ],
        'negative_influence': [
            {'factor': k, 'correlation': round(float(v), 5)} for k, v in negative.items()
        ]
    }

payload = {
    'top_pollutant': top_pollutant,
    'city_count': len(city_correlations),
    'correlations': city_correlations
}

correlation_path = ARTIFACT_DIR / 'pollutant_correlations.json'
tmp_correlation_path = correlation_path.with_suffix(correlation_path.suffix + '.tmp')

try:
    with open(tmp_correlation_path, 'w', encoding='utf-8') as fp:
        json.dump(payload, fp, indent=2)
    tmp_correlation_path.replace(correlation_path)
finally:
    if tmp_correlation_path.exists():
        tmp_correlation_path.unlink()

payload['city_count']

1687